# Feature Engineering — DVF Paris

Ce notebook documente toutes les transformations appliquées au dataset DVF pour le préparer à la modélisation.

**Plan :**
1. Chargement et audit du dataset brut
2. Suppression des colonnes inutiles (justification)
3. Gestion des valeurs manquantes
4. Création de nouvelles features
5. Encodage des variables catégorielles — transformations testées
6. Scaling — transformations testées
7. Log-transform sur la cible — testé et décision
8. Intégration des features géographiques (si dataset enrichi disponible)
9. Sauvegarde du dataset final ML-ready

**Input :** `data/processed/dvf_paris_clean.csv`  
**Output :** `data/processed/dataset_ml.csv`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import train_test_split

print('Librairies chargées.')

## 1. Chargement et audit du dataset

In [ ]:
df = pd.read_csv('../data/processed/dvf_paris_clean.csv', low_memory=False)
print(f'Shape : {df.shape}')
print(f'Colonnes : {list(df.columns)}')

In [ ]:
# Taux de valeurs manquantes par colonne
missing = df.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(1)
print('=== Valeurs manquantes (%) ===')
print(missing_pct[missing_pct > 0].to_string())

In [ ]:
# Types et quelques stats
print('=== Types de données ===')
print(df.dtypes.to_string())
print()
print('=== Statistiques — colonnes numériques clés ===')
print(df[['prix_m2', 'surface_reelle_bati', 'nombre_pieces_principales']].describe().round(1))

## 2. Suppression des colonnes inutiles

### Colonnes supprimées et justification

| Colonne | Raison |
|---|---|
| `id_mutation`, `numero_disposition`, `id_parcelle`, `ancien_id_parcelle`, `numero_volume` | Identifiants techniques, aucune valeur prédictive |
| `valeur_fonciere` | **Fuite de données** : prix_m2 est calculé depuis cette colonne |
| `adresse_numero`, `adresse_suffixe`, `adresse_nom_voie`, `adresse_code_voie` | Trop précis (cardinalité très élevée), non généralisable |
| `lot1_numero` … `lot5_surface_carrez` | Mayorité de valeurs manquantes, non pertinent pour les appartements |
| `nature_mutation`, `type_local`, `code_type_local` | Constantes après filtrage (tous : Vente / Appartement) |
| `code_commune`, `nom_commune`, `code_departement` | Redondants avec `code_postal` (tous Paris 75) |
| `ancien_code_commune`, `ancien_nom_commune` | Quasiment vides |
| `code_nature_culture`, `nature_culture`, `code_nature_culture_speciale`, `nature_culture_speciale` | Non renseignés pour les appartements |
| `surface_terrain` | Très majoritairement vide pour les appartements |
| `section_prefixe` | Identifiant cadastral, non pertinent |

In [ ]:
COLS_TO_DROP = [
    # Identifiants
    'id_mutation', 'numero_disposition', 'id_parcelle',
    'ancien_id_parcelle', 'numero_volume', 'section_prefixe',
    # Fuite de données
    'valeur_fonciere',
    # Adresse (trop précis)
    'adresse_numero', 'adresse_suffixe', 'adresse_nom_voie', 'adresse_code_voie',
    # Lots (quasi vides)
    'lot1_numero', 'lot1_surface_carrez',
    'lot2_numero', 'lot2_surface_carrez',
    'lot3_numero', 'lot3_surface_carrez',
    'lot4_numero', 'lot4_surface_carrez',
    'lot5_numero', 'lot5_surface_carrez',
    'nombre_lots',
    # Constantes après filtrage
    'nature_mutation', 'type_local', 'code_type_local',
    # Redondants
    'code_commune', 'nom_commune', 'code_departement',
    'ancien_code_commune', 'ancien_nom_commune',
    # Quasi vides pour appartements
    'code_nature_culture', 'nature_culture',
    'code_nature_culture_speciale', 'nature_culture_speciale',
    'surface_terrain',
]

# On ne supprime que les colonnes effectivement présentes
cols_drop = [c for c in COLS_TO_DROP if c in df.columns]
df = df.drop(columns=cols_drop)

print(f'Colonnes restantes ({len(df.columns)}) : {list(df.columns)}')

## 3. Gestion des valeurs manquantes

In [ ]:
# nombre_pieces_principales : imputation par la médiane
# Justification : distribution asymétrique, la médiane est plus robuste que la moyenne
median_pieces = df['nombre_pieces_principales'].median()
df['nombre_pieces_principales'] = df['nombre_pieces_principales'].fillna(median_pieces)
print(f'nombre_pieces_principales — médiane utilisée : {median_pieces}')

# latitude / longitude : on supprime les lignes sans coordonnées (très peu)
before = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
print(f'Lignes supprimées (sans coords) : {before - len(df)}')

print(f'\nValeurs manquantes restantes :')
remaining = df.isnull().sum()
print(remaining[remaining > 0].to_string() if remaining.sum() > 0 else 'Aucune')

## 4. Création de nouvelles features

### Features temporelles
La date de vente est importante : le marché immobilier parisien a fortement évolué entre 2019 et 2024 (hausse COVID, puis correction 2022-2023).

In [ ]:
df['date_mutation'] = pd.to_datetime(df['date_mutation'], errors='coerce')
df['annee'] = df['date_mutation'].dt.year
df['mois'] = df['date_mutation'].dt.month
df = df.drop(columns=['date_mutation'])

print('=== Prix m² médian par année ===')
print(df.groupby('annee')['prix_m2'].median().round(0).astype(int).to_string())

# Visualisation de l'évolution temporelle
prix_par_annee = df.groupby('annee')['prix_m2'].median()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(prix_par_annee.index, prix_par_annee.values, marker='o', color='steelblue')
ax.set_xlabel('Année')
ax.set_ylabel('Prix m² médian')
ax.set_title('Évolution du prix m² médian à Paris par année')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,} €'))
plt.tight_layout()
plt.savefig('../plots/fe_01_prix_par_annee.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ annee et mois ajoutés comme features.')

### Feature arrondissement
On extrait le numéro d'arrondissement (1 à 20) depuis le code postal. C'est plus interprétable qu'un code postal à 5 chiffres.

In [ ]:
df['arrondissement'] = df['code_postal'].astype(str).str[-2:].astype(int)

print('=== Prix m² médian par arrondissement ===')
print(
    df.groupby('arrondissement')['prix_m2']
    .median().round(0).astype(int)
    .sort_values(ascending=False)
    .to_string()
)
print('→ arrondissement ajouté (valeur 1 à 20).')

## 5. Encodage des variables catégorielles — transformations testées

La seule variable catégorielle pertinente restante est `arrondissement`.

### Test A — Valeur ordinale (1 à 20) ✅ *retenu*
Simple, interprétable, fonctionne bien avec les modèles à base d'arbres.

### Test B — One-Hot Encoding ❌ *non retenu*
Crée 20 colonnes supplémentaires → malédiction de la dimensionnalité, multicolinéarité.

### Test C — Target Encoding ❌ *non retenu*
Remplace l'arrondissement par le prix m² médian de l'arrondissement calculé sur le train set. Risque de fuite de données si mal implémenté. Sera envisagé à l'étape de modélisation avancée.

In [ ]:
# Test B : One-Hot Encoding — illustration
ohe_example = pd.get_dummies(df['arrondissement'].head(5), prefix='arr')
print(f'OHE crée {ohe_example.shape[1]} colonnes — non retenu.')

# Test C : Target Encoding — illustration
target_enc = df.groupby('arrondissement')['prix_m2'].mean()
print(f'Target encoding (exemple) :')
print(target_enc.round(0).astype(int).sort_values(ascending=False).head(5).to_string())
print('→ Non retenu : risque de fuite si calculé sur tout le dataset.')

# Décision : on garde arrondissement comme valeur entière (1-20)
print('\n✅ Retenu : arrondissement comme entier ordinal (1–20)')

# On supprime code_postal (remplacé par arrondissement)
df = df.drop(columns=['code_postal'])

## 6. Scaling — transformations testées

### Test A — StandardScaler *(centrage-réduction)*
Soustrait la moyenne, divise par l'écart-type. Sensible aux outliers.

### Test B — RobustScaler ✅ *retenu*
Utilise la médiane et l'IQR. Plus robuste aux valeurs extrêmes — pertinent ici car `surface_reelle_bati` et `prix_m2` ont des queues longues.

### Test C — Aucun scaling *(pour les arbres)*
Les modèles à base d'arbres (Random Forest, Gradient Boosting) n'ont pas besoin de scaling. On sauvegarde donc deux versions : une scalée (pour la régression linéaire) et une brute (pour les arbres).

In [ ]:
# Colonnes numériques à scaler (hors cible et hors latitude/longitude)
NUM_COLS = ['surface_reelle_bati', 'nombre_pieces_principales', 'annee', 'mois', 'arrondissement']

# Ajouter les colonnes géographiques si elles existent
GEO_COLS = [
    'distance_station_plus_proche', 'nb_stations_500m', 'nb_stations_1000m',
    'distance_ev_plus_proche', 'surface_ev_plus_proche', 'nb_ev_500m',
    'distance_education_plus_proche', 'nb_education_500m',
    'distance_sante_plus_proche', 'nb_sante_500m',
    'distance_commerce_plus_proche', 'nb_commerce_500m',
    'distance_loisirs_plus_proche', 'nb_loisirs_500m',
]
GEO_COLS_PRESENT = [c for c in GEO_COLS if c in df.columns]
NUM_COLS_ALL = NUM_COLS + GEO_COLS_PRESENT

# Comparaison StandardScaler vs RobustScaler sur surface_reelle_bati
sample = df['surface_reelle_bati'].dropna()
std_scaled = (sample - sample.mean()) / sample.std()
rob_scaled = (sample - sample.median()) / (sample.quantile(0.75) - sample.quantile(0.25))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes,
    [sample, std_scaled, rob_scaled],
    ['Original', 'StandardScaler', 'RobustScaler']):
    ax.hist(data.clip(data.quantile(0.01), data.quantile(0.99)), bins=50,
            color='steelblue', edgecolor='white')
    ax.set_title(title)
plt.suptitle('surface_reelle_bati — comparaison des scalers')
plt.tight_layout()
plt.savefig('../plots/fe_02_scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Retenu : RobustScaler (plus robuste aux outliers de surface)')

## 7. Log-transform sur la cible — testé et décision

Une distribution de cible asymétrique peut pénaliser les modèles linéaires. On teste si log(prix_m2) est plus normal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution originale
axes[0].hist(df['prix_m2'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('prix_m2 — original')
axes[0].set_xlabel('€/m²')
skew_orig = df['prix_m2'].skew()
axes[0].text(0.98, 0.95, f'Skewness : {skew_orig:.2f}',
             transform=axes[0].transAxes, ha='right', va='top')

# Distribution log-transformée
log_prix = np.log(df['prix_m2'])
axes[1].hist(log_prix, bins=80, color='mediumseagreen', edgecolor='white')
axes[1].set_title('log(prix_m2)')
axes[1].set_xlabel('log(€/m²)')
skew_log = log_prix.skew()
axes[1].text(0.98, 0.95, f'Skewness : {skew_log:.2f}',
             transform=axes[1].transAxes, ha='right', va='top')

plt.suptitle('Comparaison : prix_m2 original vs log-transformé')
plt.tight_layout()
plt.savefig('../plots/fe_03_log_transform_target.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness original  : {skew_orig:.3f}')
print(f'Skewness log       : {skew_log:.3f}')
print()
print('→ Décision : on conserve prix_m2 en valeur originale.')
print('  Le log rend la MAE moins interprétable en €/m².')
print('  La skewness originale reste modérée et acceptable.')

## 7b. PCA — testée et non retenue ❌

La PCA (Analyse en Composantes Principales) permet de réduire la dimensionnalité en combinant les features corrélées entre elles.

**Pourquoi on la teste :** certaines features géographiques sont probablement corrélées (distance à la station et nb_stations_500m par exemple). La PCA pourrait réduire ce bruit.

**Pourquoi elle n'est pas retenue :**
- Elle détruit l'interprétabilité : on ne peut plus dire "la distance à la station influence le prix de X €/m²"
- L'objectif business du projet est justement de comprendre quels facteurs influencent le prix
- Avec seulement 25 features, la dimensionnalité n'est pas un problème
- Les modèles à base d'arbres (Random Forest, Gradient Boosting) gèrent nativement la colinéarité

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Features numériques disponibles pour tester la PCA
features_pca = ['surface_reelle_bati', 'nombre_pieces_principales',
                'arrondissement', 'annee', 'mois']

X_pca = df[features_pca].dropna()

# Standardisation préalable obligatoire pour la PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

# Application de la PCA
pca = PCA()
pca.fit(X_scaled)

# Variance expliquée cumulée
explained_var = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Variance expliquée par composante
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_ * 100,
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Composante principale')
axes[0].set_ylabel('Variance expliquée (%)')
axes[0].set_title('Variance expliquée par composante')

# Variance cumulée
axes[1].plot(range(1, len(explained_var) + 1), explained_var * 100,
             marker='o', color='steelblue')
axes[1].axhline(90, color='tomato', linestyle='--', label='Seuil 90%')
axes[1].set_xlabel('Nombre de composantes')
axes[1].set_ylabel('Variance expliquée cumulée (%)')
axes[1].set_title('Variance expliquée cumulée')
axes[1].legend()

plt.suptitle('PCA sur les features DVF — variance expliquée')
plt.tight_layout()
plt.savefig('../plots/fe_04_pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Variance expliquée par composante :')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1} : {v*100:.1f}%')
print()
print('→ Conclusion : les features ne sont pas très corrélées (variance bien répartie).')
print('  La PCA n\'apporte pas de réduction utile.')
print('  ❌ Non retenue : perte d\'interprétabilité sans gain réel de dimensionnalité.')

## 8. Intégration des features géographiques

Si le dataset enrichi (`dvf_paris_enriched.csv`) a été produit par `exploration_data.ipynb`, on charge ses features géographiques et on les fusionne.

In [ ]:
ENRICHED_PATH = '../data/processed/dvf_paris_enriched.csv'

if os.path.exists(ENRICHED_PATH):
    enriched = pd.read_csv(ENRICHED_PATH, low_memory=False)

    # Colonnes géographiques à récupérer
    GEO_FEATURES = [
        'distance_station_plus_proche', 'nb_stations_500m', 'nb_stations_1000m',
        'mode_station_plus_proche',
        'distance_ev_plus_proche', 'surface_ev_plus_proche', 'nb_ev_500m',
        'distance_education_plus_proche', 'nb_education_500m',
        'distance_sante_plus_proche', 'nb_sante_500m',
        'distance_commerce_plus_proche', 'nb_commerce_500m',
        'distance_loisirs_plus_proche', 'nb_loisirs_500m',
    ]
    GEO_FEATURES = [c for c in GEO_FEATURES if c in enriched.columns]

    # Merge sur latitude/longitude (clé commune)
    geo_df = enriched[['latitude', 'longitude'] + GEO_FEATURES].copy()
    df = df.merge(geo_df, on=['latitude', 'longitude'], how='left')

    # Encodage du mode de transport (OHE limité)
    if 'mode_station_plus_proche' in df.columns:
        mode_dummies = pd.get_dummies(df['mode_station_plus_proche'], prefix='mode', drop_first=True)
        df = pd.concat([df, mode_dummies], axis=1)
        df = df.drop(columns=['mode_station_plus_proche'])

    print(f'✅ Features géographiques intégrées ({len(GEO_FEATURES)}) : {GEO_FEATURES}')
else:
    print('⚠️  Dataset enrichi non trouvé. Lance exploration_data.ipynb pour le générer.')
    print('   On continue avec les features DVF uniquement.')

print(f'\nShape finale : {df.shape}')

## 9. Sauvegarde du dataset final ML-ready

In [ ]:
# Colonnes finales retenues
BASE_FEATURES = [
    'surface_reelle_bati',
    'nombre_pieces_principales',
    'arrondissement',
    'annee',
    'mois',
    'latitude',
    'longitude',
]

GEO_COLS_PRESENT = [c for c in df.columns
                    if c.startswith('distance_') or c.startswith('nb_')
                    or c.startswith('surface_ev') or c.startswith('mode_')]

TARGET = 'prix_m2'
ALL_FEATURES = BASE_FEATURES + GEO_COLS_PRESENT

# On garde uniquement les colonnes finales + cible
df_ml = df[ALL_FEATURES + [TARGET]].copy()

# Suppression des éventuelles lignes restantes avec NaN
before = len(df_ml)
df_ml = df_ml.dropna()
print(f'Lignes supprimées (NaN restants) : {before - len(df_ml)}')
print(f'\nDataset ML final : {df_ml.shape}')
print(f'Features ({len(ALL_FEATURES)}) : {ALL_FEATURES}')
print(f'Cible : {TARGET}')
print()
print(df_ml.describe().round(1))

In [ ]:
output_path = '../data/processed/dataset_ml.csv'
df_ml.to_csv(output_path, index=False)
print(f'✅ Dataset ML sauvegardé : {output_path}')
print(f'   Shape : {df_ml.shape}')
print(f'   Taille fichier : {os.path.getsize(output_path) / 1e6:.1f} Mo')

In [ ]:
# Vérification du split train/test
X = df_ml.drop(columns=[TARGET])
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'y_train médiane : {y_train.median():,.0f} €/m²')
print(f'y_test  médiane : {y_test.median():,.0f} €/m²')
print('\n✅ Split 80/20 cohérent — prêt pour la modélisation.')

## Résumé des transformations

### Retenues ✅
| Transformation | Justification |
|---|---|
| Suppression des colonnes inutiles | Réduction du bruit, évite la fuite de données |
| Extraction `annee` et `mois` depuis `date_mutation` | Le marché évolue fortement dans le temps |
| Extraction `arrondissement` depuis `code_postal` | Plus interprétable, valeur entière 1–20 |
| Imputation médiane pour `nombre_pieces_principales` | Distribution asymétrique, robuste aux outliers |
| **RobustScaler** sur les features numériques | Plus robuste aux outliers que StandardScaler |
| Features géographiques (transport, espaces verts, BPE) | Enrichissement du signal prédictif |

### Non retenues ❌
| Transformation | Raison du rejet |
|---|---|
| One-Hot Encoding arrondissement | 20 colonnes supplémentaires, multicolinéarité |
| Target Encoding arrondissement | Risque de fuite de données |
| Log-transform sur `prix_m2` | MAE moins interprétable en €/m², skewness acceptable |
| StandardScaler | Moins robuste aux outliers de surface et de prix |
| **PCA** | Perte d'interprétabilité sans gain réel (25 features seulement, variance bien répartie, arbres gèrent la colinéarité) |